In [4]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# -------------------------------
# 📂 PATHS
# -------------------------------
train_dir = r'D:\Internship-Project\MLprojects\ml_face_detection\archive (1)\rvf10k\train'
valid_dir = r'D:\Internship-Project\MLprojects\ml_face_detection\archive (1)\rvf10k\valid'

# -------------------------------
# IMAGE GENERATORS
# -------------------------------
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.4,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.5, 1.5]
)

valid_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

valid_generator = valid_datagen.flow_from_directory(
    valid_dir,
    target_size=(128, 128),
    batch_size=32,
    class_mode='binary'
)

# 🔥 IMPORTANT: Check class mapping
print("Class Indices:", train_generator.class_indices)

# -------------------------------
# MODEL (MobileNetV2)
# -------------------------------
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128, 128, 3)
)

# Freeze base layers
for layer in base_model.layers:
    layer.trainable = False

# Custom head
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# -------------------------------
# CALLBACKS
# -------------------------------
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    r'D:\Internship-Project\MLprojects\ml_face_detection\best_model.keras',
    monitor='val_accuracy',
    save_best_only=True
)

# -------------------------------
# TRAINING (PHASE 1)
# -------------------------------
history = model.fit(
    train_generator,
    epochs=25,
    validation_data=valid_generator,
    callbacks=[early_stop, checkpoint]
)

# -------------------------------
# FINE-TUNING (PHASE 2)
# -------------------------------
for layer in base_model.layers[-30:]:
    layer.trainable = True

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    epochs=10,
    validation_data=valid_generator
)

# -------------------------------
# SAVE FINAL MODEL
# -------------------------------
model.save(r'D:\Internship-Project\MLprojects\ml_face_detection\deepfake_model.keras')

print("✅ Training Complete & Model Saved!")

Found 7000 images belonging to 2 classes.
Found 3000 images belonging to 2 classes.
Class Indices: {'fake': 0, 'real': 1}
Epoch 1/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 169s 758ms/step - accuracy: 0.5491 - loss: 0.7580 - val_accuracy: 0.5893 - val_loss: 0.6744
Epoch 2/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 52s 238ms/step - accuracy: 0.5763 - loss: 0.6847 - val_accuracy: 0.6213 - val_loss: 0.6524
Epoch 3/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 55s 249ms/step - accuracy: 0.6120 - loss: 0.6588 - val_accuracy: 0.6337 - val_loss: 0.6438
Epoch 4/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 58s 264ms/step - accuracy: 0.6104 - loss: 0.6549 - val_accuracy: 0.6153 - val_loss: 0.6504
Epoch 5/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 54s 247ms/step - accuracy: 0.6183 - loss: 0.6486 - val_accuracy: 0.6630 - val_loss: 0.6262
Epoch 6/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 53s 241ms/step - accuracy: 0.6281 - loss: 0.6483 - val_accuracy: 0.6603 - val_loss: 0.6221
Epoch 7/25
219/219 ━━━━━━━━━━━━━━━━━━━━ 50s 226ms/step - accuracy: 0.6307 - loss: 0.6408 - 

In [5]:
print("Model Input Shape:", model.input_shape)

Model Input Shape: (None, 128, 128, 3)
